# Notebook 03 — Regresión Mincer separada por sexo

Tres objetivos:

1. Mostrar cómo cambia la "brecha residual" al ir agregando controles uno por uno (modelos M0–M3 con dummy de mujer).
2. Verificar el sanity check del retorno a la escolaridad contra la literatura mexicana.
3. Estimar el modelo final separado por sexo — la entrada a Oaxaca-Blinder en el notebook 04.

Todo con WLS ponderado por `fac_tri` y errores robustos HC3. La variable dependiente es `log_ingreso_mensual_w` (winsorizada).


In [ ]:
from pathlib import Path
import sys, json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import enoe_loader as enoe

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 160, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
})
COLOR_H = "#2c7fb8"; COLOR_M = "#d95f0e"
OUTPUTS = ROOT / "outputs"

df = pd.read_parquet(ROOT / "data" / "processed" / "enoen_2025_4t_mincer.parquet")

# Variables categóricas como strings para que C() las trate bien
df["sector_cat"]    = df["sector"].astype(int).astype(str)
df["ent_cat"]       = df["ent"].astype(int).astype(str)
df["subordinado_i"] = df["subordinado"].astype(int)

print(f"Dataset: {len(df):,} obs")
print(f"  Hombres: {(df['mujer']==0).sum():,}")
print(f"  Mujeres: {(df['mujer']==1).sum():,}")


## 1. Modelos escalados: cuánto explican los controles

Empiezo con un modelo pooled (hombres y mujeres juntos) que incluye una dummy `mujer`. El coeficiente de esa dummy es la brecha residual del log-ingreso mensual controlando por lo que tenga el modelo. Voy agregando controles uno por uno para ver cómo cae:

- **M0:** solo horas y mujer
- **M1:** + escolaridad, experiencia, experiencia² (Mincer puro)
- **M2:** + sector y condición de subordinado
- **M3:** + entidad federativa (especificación completa)


In [ ]:
modelos = {
    "M0 — solo horas y mujer":
        "log_ingreso_mensual_w ~ mujer + log_horas_w",
    "M1 — Mincer ampliado":
        "log_ingreso_mensual_w ~ mujer + log_horas_w + anios_escolaridad + experiencia + experiencia2",
    "M2 — + sector y subordinado":
        "log_ingreso_mensual_w ~ mujer + log_horas_w + anios_escolaridad + experiencia + experiencia2 + C(sector_cat) + subordinado_i",
    "M3 — + entidad federativa":
        "log_ingreso_mensual_w ~ mujer + log_horas_w + anios_escolaridad + experiencia + experiencia2 + C(sector_cat) + subordinado_i + C(ent_cat)",
}

filas = []
for nombre, formula in modelos.items():
    res = smf.wls(formula, data=df, weights=df["fac_tri"]).fit(cov_type="HC3")
    filas.append({
        "Modelo": nombre,
        "R²": res.rsquared,
        "β mujer": res.params["mujer"],
        "% brecha residual": (np.exp(res.params["mujer"]) - 1) * 100,
    })
pd.DataFrame(filas).set_index("Modelo").round(4)


**Lo que la tabla muestra:**

Agregar controles uno por uno reduce un poco la brecha residual, pero no la cierra. Pasa de −19.9% (M1) a −19.5% (M3). El R² sí mejora bastante (0.38 → 0.50 al agregar sector, formalidad y entidad).

Conclusión: **la brecha mensual del 23% NO se explica con las variables observables clásicas**. Aún igualando horas trabajadas, escolaridad, experiencia, sector, condición de subordinado y entidad, las mujeres ganan 17–19% menos por mes. Esa es la brecha "residual" preliminar.


## 2. Sanity check: retorno a escolaridad clásico

El modelo Mincer estándar va sobre `log(ingreso por hora)` (sin `log_horas` como control, porque eso ya está implícito al dividir). El coeficiente de `anios_escolaridad` ahí es el retorno por año de escolaridad, y debe estar en línea con la literatura mexicana.

Cifras de referencia:
- Campos Vázquez et al. (2022): 8–10% por año en datos 2018
- BANXICO (2024): retornos en torno a 6–8% para la generación más reciente, mostrando caída secular
- INEGI publicaciones más recientes: 5–7% en ciertas submuestras

Espero algo entre 5% y 10%.


In [ ]:
formula_clasico = ("log_ingreso_hora_w ~ mujer + anios_escolaridad + experiencia + experiencia2 "
                   "+ C(sector_cat) + subordinado_i + C(ent_cat)")
res_clasico = smf.wls(formula_clasico, data=df, weights=df["fac_tri"]).fit(cov_type="HC3")

print(f"R² = {res_clasico.rsquared:.4f}")
print(f"\nRetorno a escolaridad: {res_clasico.params['anios_escolaridad']*100:.2f}% por año")
print(f"Brecha residual por hora (β mujer): {(np.exp(res_clasico.params['mujer'])-1)*100:+.1f}%")


**Notas importantes:**

El retorno a escolaridad sale en ~5.1% por año, dentro del rango bajo de la literatura mexicana reciente. La caída secular de los retornos a la educación en México (Lustig, Esquivel, Campos) está documentada: más oferta de educados sin igual aumento de demanda → retornos cayendo.

Más interesante: la brecha residual por hora (−7.5%) es **mayor** que la brecha bruta por hora (1.9%, ver notebook 02). Esto es supresión estadística: las mujeres tienen más escolaridad y más probabilidad de ser asalariadas, así que "deberían" ganar más por hora. Que no lo hagan deja un residual mayor. Es un hallazgo robusto, no un artefacto.


## 3. Modelo separado por sexo (entrada para Oaxaca-Blinder)

Lo que sí va al notebook 04. Misma fórmula M3, pero estimada por separado para hombres y mujeres. La diferencia en coeficientes entre los dos modelos es exactamente lo que Oaxaca-Blinder llama "componente de coeficientes" — la parte de la brecha que no se explica por diferencias en características, sino por diferencias en retornos.


In [ ]:
formula = ("log_ingreso_mensual_w ~ log_horas_w + anios_escolaridad + experiencia + experiencia2 "
           "+ C(sector_cat) + subordinado_i + C(ent_cat)")

h = df[df["mujer"]==0]
m = df[df["mujer"]==1]

res_h = smf.wls(formula, data=h, weights=h["fac_tri"]).fit(cov_type="HC3")
res_m = smf.wls(formula, data=m, weights=m["fac_tri"]).fit(cov_type="HC3")

print(f"R² Hombres: {res_h.rsquared:.4f}  ·  n = {int(res_h.nobs):,}")
print(f"R² Mujeres: {res_m.rsquared:.4f}  ·  n = {int(res_m.nobs):,}")
print(f"\nIntercept Hombres: {res_h.params['Intercept']:+.4f}")
print(f"Intercept Mujeres: {res_m.params['Intercept']:+.4f}")
print(f"Diferencia intercept: {res_h.params['Intercept']-res_m.params['Intercept']:+.4f}")


In [ ]:
variables = {
    "log_horas_w":       "log(horas) — elasticidad",
    "anios_escolaridad": "Años de escolaridad",
    "experiencia":       "Experiencia",
    "experiencia2":      "Experiencia²",
    "subordinado_i":     "Es asalariado",
}

filas = []
for var, etiqueta in variables.items():
    bh, seh = res_h.params[var], res_h.bse[var]
    bm, sem = res_m.params[var], res_m.bse[var]
    z = (bh - bm) / np.sqrt(seh**2 + sem**2)
    sig = "***" if abs(z)>2.58 else ("**" if abs(z)>1.96 else ("*" if abs(z)>1.65 else ""))
    filas.append({
        "Variable": etiqueta,
        "β H": f"{bh:+.4f}", "(SE H)": f"({seh:.4f})",
        "β M": f"{bm:+.4f}", "(SE M)": f"({sem:.4f})",
        "Diff": f"{bh-bm:+.4f}", "z": f"{z:+.2f}", "Sig": sig,
    })
pd.DataFrame(filas)


**Lo que la tabla muestra:**

Todas las diferencias entre coeficientes son altamente significativas (z grande en valor absoluto). En particular:

- **Elasticidad de ingreso a horas: H=0.46, M=0.55**. Las mujeres "convierten" mejor el aumento de horas en ingreso, probablemente porque están más en empleos por hora (comercio, retail, doméstico) donde la relación es más directa.
- **Retorno a escolaridad: H=3.9%, M=5.1%**. Las mujeres obtienen más por cada año adicional de escuela. Pero ya en promedio tienen más educación; lo que limita su ingreso no es la palanca educativa.
- **Experiencia: H=1.9%, M=1.4%**. Los hombres ven más retorno por experiencia. Probable proxy de tiempo continuo en el mercado (las mujeres tienen interrupciones por cuidados que no se capturan en "experiencia potencial").
- **Asalariado: H=−0.24, M=−0.36**. Ser asalariada penaliza más el ingreso de mujeres que de hombres. Curioso y vale la pena explorar después.

El **intercept** difiere por +0.59 a favor de hombres. Con todas las variables iguales a sus referencias (sector base, entidad base, cero escolaridad, cero experiencia, no asalariado), un hombre parte de un nivel base más alto. Pero ese coeficiente solo no es "discriminación pura" — depende de cómo se definan las categorías de referencia. La medida correcta la da Oaxaca-Blinder.


## 4. Forest plot — comparación visual de coeficientes


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
y_pos = np.arange(len(variables))[::-1]
offset = 0.18

for i, (var, etiqueta) in enumerate(variables.items()):
    y = y_pos[i]
    bh, seh = res_h.params[var], res_h.bse[var]
    bm, sem = res_m.params[var], res_m.bse[var]
    ax.errorbar(bh, y+offset, xerr=1.96*seh, fmt="o", color=COLOR_H,
                capsize=4, markersize=8, label="Hombres" if i==0 else None)
    ax.errorbar(bm, y-offset, xerr=1.96*sem, fmt="o", color=COLOR_M,
                capsize=4, markersize=8, label="Mujeres" if i==0 else None)
    ax.text(bh, y+offset+0.16, f"{bh:+.3f}", ha="center", fontsize=8.5, color=COLOR_H)
    ax.text(bm, y-offset-0.22, f"{bm:+.3f}", ha="center", fontsize=8.5, color=COLOR_M)

ax.axvline(0, color="black", linestyle="-", linewidth=0.5, alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(list(variables.values()))
ax.set_xlabel("Coeficiente (efecto sobre log ingreso mensual)")
ax.set_title("Hombres y mujeres tienen retornos distintos a las mismas características")
ax.text(0.99, -0.14, "Mincer WLS ponderado, errores HC3. IC 95%. ENOE 4T 2025.",
        transform=ax.transAxes, ha="right", fontsize=8, color="gray")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUTS / "mincer_g1_coeficientes.png")
plt.show()


## 5. Guardar resultados para el notebook 04

Statsmodels con fórmulas no se serializa bien con pickle entre procesos. Guardo `params` y `bse` en JSON; el notebook 04 reconstruye lo que necesite.


In [ ]:
tabla = pd.DataFrame({
    "variable": list(variables.keys()),
    "etiqueta": list(variables.values()),
    "beta_hombres": [res_h.params[v] for v in variables],
    "se_hombres":   [res_h.bse[v]    for v in variables],
    "beta_mujeres": [res_m.params[v] for v in variables],
    "se_mujeres":   [res_m.bse[v]    for v in variables],
})
tabla["diferencia"] = tabla["beta_hombres"] - tabla["beta_mujeres"]
tabla.to_csv(OUTPUTS / "mincer_coeficientes_por_sexo.csv", index=False)

with open(OUTPUTS / "mincer_params.json", "w") as f:
    json.dump({
        "hombres": {"params": res_h.params.to_dict(), "bse": res_h.bse.to_dict(),
                    "r2": res_h.rsquared, "n": int(res_h.nobs)},
        "mujeres": {"params": res_m.params.to_dict(), "bse": res_m.bse.to_dict(),
                    "r2": res_m.rsquared, "n": int(res_m.nobs)},
        "formula": formula,
    }, f, indent=2)

print("Archivos generados:")
print("  outputs/mincer_g1_coeficientes.png")
print("  outputs/mincer_coeficientes_por_sexo.csv")
print("  outputs/mincer_params.json")


## Debrief del notebook 03

**Lo que aprendí:**

1. La brecha mensual residual ronda 17–19% después de controlar por horas, escolaridad, experiencia, sector, subordinación y entidad. Los controles bajan el R² del 0.25 al 0.50, pero apenas mueven el coeficiente de "mujer". Eso es señal fuerte de que la brecha no se explica con lo observable en ENOE.

2. La brecha residual por hora (7.5%) es **mayor** que la brecha bruta por hora (1.9%). Supresión: las mujeres tienen más educación y más subordinación, "deberían" ganar más por hora pero no lo hacen.

3. Los coeficientes difieren significativamente entre sexos en todas las variables clave. Eso garantiza que Oaxaca-Blinder va a encontrar un componente de "no explicado" sustantivo.

**Lo que se me resistió:**

- Statsmodels con fórmulas tipo patsy NO se serializa entre procesos de Python — los modelos guardados con pickle tronaban al re-cargarlos. Lección: para colaborar entre notebooks, guardar `params.to_dict()` y `bse.to_dict()` en JSON, no el objeto entero.
- La etiqueta "formal" en mi loader inicial era engañosa. `emp_ppal == 1` en ENOE codifica "subordinado y remunerado", no "formalidad en sentido estricto". Renombré la variable a `subordinado` para no engañar al lector.
- El retorno a escolaridad de 5% asustó a primera vista. Tuve que verificar contra literatura mexicana reciente — sí está dentro del rango actual, aunque más bajo que las cifras "icónicas" de 8-10% que reportan estudios viejos.

**Para el notebook 04 (Oaxaca-Blinder):**

- Cargar `mincer_params.json` y re-ajustar los modelos H/M en código limpio.
- Calcular descomposición Oaxaca-Blinder con la versión pooled (Neumark) como principal y reportar las otras dos (Oaxaca clásica con referencia hombre y mujer) como anexo.
- Bootstrap con 1000 réplicas para IC 95%.
- Tres bloques en la descomposición: aporte de **horas**, aporte de **composición** (sector + escolaridad + experiencia + subordinación + entidad), y aporte **residual** (parte no explicada).
